### Step 1. Abril - Junio - Agosto (FE04b)


#### Librerías y Path Variables

In [1]:
# Configuración de recursos según la máquina
entorno = 'oficina'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

Entorno: oficina | DuckDB: 12 threads, 45GB | RF: n_jobs=10 | LGB: 10 threads


In [2]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt

from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

from sklearn.model_selection import StratifiedShuffleSplit

import lightgbm as lgb

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour


In [3]:
# Paths Variables

from pathlib import Path
cwd = Path.cwd()
if cwd.name == 'scripts':
    RAIZ = cwd.parent
elif cwd.name == '03_competencia_01':
    RAIZ = cwd
else:
    RAIZ = cwd / '03_competencia_01'

sys.path.insert(0, str(RAIZ / 'scripts'))
from semillas import SEMILLAS as semillas

base_path = str(RAIZ) + '/'
print(RAIZ, RAIZ.exists(), (RAIZ / 'scripts' / 'semillas.py').exists())

dataset_path = base_path + 'data/'
modelos_path = base_path + 'modelos/'
db_path = base_path + 'data/databases/'
dataset_file = 'competencia_01_fe03.parquet'

d:\Training\05_Exactas-UBA\06_DM_EconomiaFinanzas\03_competencia_01 True True


#### Loading Data and Input Variables

##### Cargado de Datos

In [4]:
# Función para cargar los datos de la competencia, filtrando por meses y columnas

ruta_fe03 = dataset_path + dataset_file       # .../data/competencia_01_fe03.parquet
ROTA = 'ccajas_depositos'
ROTA_CONFIRMADA = True                        # resultado del chequeo de la otra notebook
con = duckdb.connect()

def cargar(archivo_cols, meses):
    cols = Path(archivo_cols).read_text().split('\n')          # nombres de columnas, desde el .txt
    select_cols = ', '.join(f'"{c}"' for c in cols)            # "col1", "col2", ... para el SELECT
    lista_meses = ', '.join(str(m) for m in meses)             # 202103, 202104, ... para el WHERE

    df = con.execute(f"""
        select {select_cols} from read_parquet('{ruta_fe03}')
        where foto_mes in ({lista_meses})
    """).fetchdf()

  # la variable rota: NaN solo en mayo
    if ROTA_CONFIRMADA and ROTA in df.columns:
        df.loc[df['foto_mes'] == 202105, ROTA] = np.nan

    return df

In [5]:
# Cargar los datos de la competencia, filtrando por meses y columnas

MESES = [202103, 202104, 202105, 202106, 202108]

OPCION = 'b'      # '' = sin históricas | 'b' = con delta_1

archivos = {
    'a': dataset_path  + 'fe04_a_sin_historicas.txt',
    'b': dataset_path  + 'fe04_b_con_delta1.txt',
}

data = cargar(archivos[OPCION], MESES)

data = data.sort_values(['foto_mes', 'numero_de_cliente'])      # CAMBIO: orden fijo de filas

print(data.shape)
print(data['foto_mes'].value_counts().sort_index())   # filas por mes

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(818713, 602)
foto_mes
202103    162900
202104    163284
202105    163768
202106    164114
202108    164647
Name: count, dtype: int64


##### Ganancia/Costo y Abril/Junio para Train/Test 

In [6]:
# Ganancia y Costo y Selección de Meses para Entrenamiento y Test

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202104    # Abril - mes de entrenamiento
mes_test = 202106     # Junio - mes de testeo

##### Clase Binaria + X/Y_Train/Test

In [7]:
data['clase_binaria_b2'] = np.where(data['clase_ternaria'] == 'BAJA+2', 1, 0)    # optimizo y mido con esta


# Preparación de datos para la optimización
cols_descartar = ['clase_ternaria', 'clase_binaria_b2', 'numero_de_cliente', 'foto_mes']

train_data = data[data['foto_mes'] == mes_train]
test_data  = data[data['foto_mes'] == mes_test]

X_train = train_data.drop(columns=cols_descartar)
y_train_binaria = train_data['clase_binaria_b2']

X_test = test_data.drop(columns=cols_descartar)
y_test_binaria = test_data['clase_binaria_b2']        # ← era 'clase_binaria'
y_test_class = test_data['clase_ternaria']            # solo para medir; no entra en X

train_data_lgb = lgb.Dataset(X_train, label=y_train_binaria)
test_data_lgb  = lgb.Dataset(X_test, label=y_test_binaria, reference=train_data_lgb)

#### Optimización de Hiperparámetros (con datos abril y cross-validation)

In [8]:
sss_opt = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=semillas[0])   # stratified shuffle split para la optimización bayesiana 
                                                                                        # con 5 splits y 30% de testeo

def objective(trial):

    # bayesianos
    num_leaves = trial.suggest_int('num_leaves', 8, 256)                                # numero de hojas del árbol
    min_sum_hessian_in_leaf = trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 100, log=True)
    learning_rate = trial.suggest_float('learning_rate', 0.005, 0.2, log=True)          # mas bajo, más iteraciones necesita
    feature_fraction = trial.suggest_float('feature_fraction', 0.3, 0.7)                # columnas por árbol
    feature_fraction_bynode = trial.suggest_float('feature_fraction_bynode', 0.1, 0.5)  # columnas por nodo (se multiplica con la anterior)
    bagging_fraction = trial.suggest_float('bagging_fraction', 0.6, 1.0)                # submuestreo de filas por árbol
    bagging_freq = trial.suggest_int('bagging_freq', 1, 10)                             # > 0 para que el bagging funcione

    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'boost_from_average': True,
        'feature_pre_filter': False,        
      # fijos de la cátedra
        'max_bin': 31,                       # número de bins para discretizar las variables continuas
        'min_data_in_leaf': 0,               # el freno pasa a min_sum_hessian_in_leaf
      # bayesianos
        'num_leaves': num_leaves,
        'min_sum_hessian_in_leaf': min_sum_hessian_in_leaf,
        'learning_rate': learning_rate,
        'feature_fraction': feature_fraction,
        'feature_fraction_bynode': feature_fraction_bynode,
        'bagging_fraction': bagging_fraction,
        'bagging_freq': bagging_freq,
        'seed': semillas[0],
        'num_threads': lgb_threads,
        'verbose': -1
    }
    num_boost_round = 3000                   # subido de 3000 porque el learning_rate mínimo es más bajo
    cv_results = lgb.cv(
        params,
        train_data_lgb,
        num_boost_round=num_boost_round,
        callbacks=[lgb.early_stopping(stopping_rounds=min(int(50 + 5 / learning_rate), num_boost_round // 3), verbose=False)],  
        folds=sss_opt.split(X_train, y_train_binaria),                    # usamos StratifiedShuffleSplit en vez del k-fold interno
        seed=semillas[0]
    )

    max_auc = max(cv_results['valid auc-mean'])
    best_iter = cv_results['valid auc-mean'].index(max_auc) + 1     # es el num_iterations

    # Guardamos cual es la mejor iteración del modelo
    trial.set_user_attr("best_iter", best_iter)

    return max_auc

storage_name = "sqlite:///" + db_path + "cpt2_optim_lgbm_Step1.db"
study_name = "cpt2_step1_b2_lgbm_sss_3000_con_FE04-b"                            # se hizo con FE04b es decir con deltas además de intrames

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)

print(len(study.trials))     # debe dar 0 si la base es nueva

[I 2026-10-10 21:34:04,278] A new study created in RDB with name: cpt2_step1_b2_lgbm_sss_3000_con_FE04-b


0


In [9]:
objetivo_total = 50                                        # número de trials que queremos correr en total
faltan = objetivo_total - len(study.trials)
print(f"Tengo {len(study.trials)}, corro {faltan} más")
if faltan > 0:
    study.optimize(objective, n_trials=faltan)

Tengo 0, corro 50 más


[I 2026-10-10 21:35:18,815] Trial 0 finished with value: 0.9045842668370897 and parameters: {'num_leaves': 69, 'min_sum_hessian_in_leaf': 0.07008535951870287, 'learning_rate': 0.037805809778033134, 'feature_fraction': 0.5336728430931723, 'feature_fraction_bynode': 0.38239167996944623, 'bagging_fraction': 0.7499682797133614, 'bagging_freq': 4}. Best is trial 0 with value: 0.9045842668370897.
[I 2026-10-10 21:37:07,064] Trial 1 finished with value: 0.9102515422948733 and parameters: {'num_leaves': 231, 'min_sum_hessian_in_leaf': 5.1299558695978, 'learning_rate': 0.01288762129673188, 'feature_fraction': 0.6618565213716294, 'feature_fraction_bynode': 0.23661536352297372, 'bagging_fraction': 0.7184597143816052, 'bagging_freq': 8}. Best is trial 1 with value: 0.9102515422948733.
[I 2026-10-10 21:40:33,588] Trial 2 finished with value: 0.906387942762094 and parameters: {'num_leaves': 216, 'min_sum_hessian_in_leaf': 0.01947398348475732, 'learning_rate': 0.012095324484235784, 'feature_fraction'

##### Resultados Optimización

In [10]:
optuna.visualization.plot_optimization_history(study)

In [11]:
plot_param_importances(study)

In [12]:
plot_slice(study)

In [13]:
plot_contour(study)

In [14]:
plot_contour(study, params=['learning_rate','min_sum_hessian_in_leaf'] )

##### Top 5 de acuerdo con AUC de la Optimización

In [26]:
# top 5 por AUC (solo trials completos) y su best_iter
def top_k(study, k):
    completos = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    return sorted(completos, key=lambda t: t.value, reverse=True)[:k]

top5 = top_k(study, 5)
for t in top5:
    print(t.number, round(t.value, 4), 'best_iter =', t.user_attrs['best_iter'])

# ¿qué pasó con el trial 11?
print(study.trials[11].state)

# ¿algún best_iter pegado al tope de 3000?
print(max(t.user_attrs['best_iter'] for t in study.trials if t.value is not None))

39 0.9118 best_iter = 884
41 0.9117 best_iter = 1168
25 0.9115 best_iter = 1052
43 0.9115 best_iter = 794
26 0.9114 best_iter = 901
TrialState.COMPLETE
2984


In [27]:
def top_k(study, k):
    completos = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    return sorted(completos, key=lambda t: t.value, reverse=True)[:k]

top5 = top_k(study, 5)

df_top5_auc = pd.DataFrame([
    {"orden": i, "trial": t.number, "auc_cv": t.value,
     "best_iter": t.user_attrs["best_iter"], **t.params}
    for i, t in enumerate(top5, start=1)
])
df_top5_auc

,orden,trial,auc_cv,best_iter,num_leaves,min_sum_hessian_in_leaf,learning_rate,feature_fraction,feature_fraction_bynode,bagging_fraction,bagging_freq
0,1,39,0.911793,884,193,5.969626,0.005357,0.623329,0.185365,0.716952,7
1,2,41,0.911666,1168,216,10.646127,0.005063,0.566551,0.199977,0.691943,8
2,3,25,0.911508,1052,220,18.110319,0.006015,0.581566,0.233214,0.791708,6
3,4,43,0.911495,794,185,8.611457,0.005693,0.637746,0.169369,0.707726,8
4,5,26,0.911425,901,191,11.217379,0.005362,0.630857,0.258448,0.804486,7


#### Validación con Datos Junio

##### Función de Ganancia

In [28]:
# Función de Ganancia
def gan_prob(y_pred, y_true, prop = 1):
  ganancia = np.where(y_true == 1, ganancia_acierto, 0) - np.where(y_true == 0, costo_estimulo, 0)
  return ganancia[y_pred >= 0.025].sum() / prop

In [18]:
top_n = 5                     # NO LA CORRO PORQUE LEVANTO LOS MODELOS DE ARCHIVO
top_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value,
    reverse=True
)[:top_n]

train_data = lgb.Dataset(X_train, label=y_train_binaria)

resultados = []

for i, trial in enumerate(top_trials, start=1):

    best_iter = trial.user_attrs["best_iter"]

    params = {
        'objective': 'binary',
        'boosting_type': 'gbdt',
        'first_metric_only': True,
        'boost_from_average': True,
        'feature_pre_filter': False,
        'max_bin': 31,
        'min_data_in_leaf': 0,                                                 # CAMBIO: fijo, ya no está en trial.params
        'num_leaves': trial.params['num_leaves'],
        'min_sum_hessian_in_leaf': trial.params['min_sum_hessian_in_leaf'],    # CAMBIO: nuevo
        'learning_rate': trial.params['learning_rate'],
        'feature_fraction': trial.params['feature_fraction'],
        'feature_fraction_bynode': trial.params['feature_fraction_bynode'],    # CAMBIO: nuevo
        'bagging_fraction': trial.params['bagging_fraction'],
        'bagging_freq': trial.params['bagging_freq'],                          # CAMBIO: nuevo
        'seed': semillas[0],
        'num_threads': lgb_threads,                                            # CAMBIO: nuevo
        'verbose': -1
    }

    model = lgb.train(params, train_data_lgb, num_boost_round=best_iter)

    y_pred_junio = model.predict(X_test)                                       # CAMBIO: mayo → junio
    ganancia_junio = gan_prob(y_pred_junio, y_test_binaria)                    # CAMBIO: mayo → junio

    resultados.append({
        "orden": i,
        "trial": trial.number,
        "valor_cv": trial.value,
        "best_iter": best_iter,
        "ganancia_junio": ganancia_junio,                                      # CAMBIO: mayo → junio
        **trial.params
    })

df_top5 = pd.DataFrame(resultados)                                             # CAMBIO: df_top3 → df_top5
df_top5

,orden,trial,valor_cv,best_iter,ganancia_junio,num_leaves,min_sum_hessian_in_leaf,learning_rate,feature_fraction,feature_fraction_bynode,bagging_fraction,bagging_freq
0,1,39,0.911793,884,335197500.0,193,5.969626,0.005357,0.623329,0.185365,0.716952,7
1,2,41,0.911666,1168,333712500.0,216,10.646127,0.005063,0.566551,0.199977,0.691943,8
2,3,25,0.911508,1052,332832500.0,220,18.110319,0.006015,0.581566,0.233214,0.791708,6
3,4,43,0.911495,794,328982500.0,185,8.611457,0.005693,0.637746,0.169369,0.707726,8
4,5,26,0.911425,901,335747500.0,191,11.217379,0.005362,0.630857,0.258448,0.804486,7


##### Incorporando Semillas 20 x 5 = 100 Modelos 

In [19]:

print(len(semillas), "semillas disponibles")      # necesito al menos 40 (20 junio + 20 agosto)

n_semillas = 20
carpeta_modelos = modelos_path + 'cpt04_modelos_TrainAbr_FE04b_opt/'                
os.makedirs(carpeta_modelos, exist_ok=True)                               # CAMBIO

train_data = lgb.Dataset(X_train, label=y_train_binaria)

rows = []
resultados_junio = {}                                                         # probabilidades de junio por (trial, semilla)

for _, fila in df_top5.iterrows():
    for seed in semillas[:n_semillas]:
        params = {
            'objective': 'binary',
            'boosting_type': 'gbdt',
            'first_metric_only': True,
            'boost_from_average': True,
            'feature_pre_filter': False,
            'max_bin': 31,
            'min_data_in_leaf': 0,
            'num_leaves': int(fila['num_leaves']),
            'min_sum_hessian_in_leaf': fila['min_sum_hessian_in_leaf'],
            'learning_rate': fila['learning_rate'],
            'feature_fraction': fila['feature_fraction'],
            'feature_fraction_bynode': fila['feature_fraction_bynode'],
            'bagging_fraction': fila['bagging_fraction'],
            'bagging_freq': int(fila['bagging_freq']),
            'seed': seed,
            'num_threads': lgb_threads,
            'verbose': -1
        }

        model = lgb.train(params, train_data, num_boost_round=int(fila['best_iter']))
        model.save_model(carpeta_modelos + f"cpt04_TrainAbr-TestlJun-FE04b_Top5Sem_mod_t{int(fila['trial'])}_s{seed}.txt")   # CAMBIO

        y_pred_junio = model.predict(X_test)
        ganancia_junio = gan_prob(y_pred_junio, y_test_binaria)

        resultados_junio[(int(fila['trial']), seed)] = y_pred_junio           # guardo las probabilidades
        rows.append({
            "trial": int(fila['trial']),
            "seed": seed,
            "ganancia_junio": ganancia_junio
        })

    # al terminar los 20 de este trial, guardo todo lo acumulado
    with open(modelos_path + 'cpt04_TrainAbr-TestlJun-FE04b_Top5Sem_prob.pkl', 'wb') as f:    # CAMBIO: nombre
        pickle.dump(resultados_junio, f)
    print(f"trial {int(fila['trial'])} guardado | {len(resultados_junio)} modelos")

np.save(modelos_path + 'cpt04_TrainAbr-TestlJun-FE04b_ids_jun.npy', data.loc[X_test.index, 'numero_de_cliente'].values)   # CAMBIO: ids de junio

df_semillas = pd.DataFrame(rows)
df_semillas.to_pickle(modelos_path + 'cpt04_TrainAbr-TestlJun-FE04b_Top5Sem_gan.pkl')      # CAMBIO: nombre

estadisticas = (
    df_semillas.groupby('trial')['ganancia_junio']
    .agg(['mean', 'std', 'min', 'max', 'median'])
    .sort_values('median', ascending=False)
)
estadisticas

100 semillas disponibles
trial 39 guardado | 20 modelos
trial 41 guardado | 40 modelos
trial 25 guardado | 60 modelos
trial 43 guardado | 80 modelos
trial 26 guardado | 100 modelos


,mean,std,min,max,median
trial,,,,,
25,337357625.0,4.239044e+06,330357500.0,344245000.0,336985000.0
41,334984375.0,4.716811e+06,327442500.0,347297500.0,334386250.0
26,332557500.0,3.860108e+06,323647500.0,339102500.0,332970000.0
43,332601500.0,5.449226e+06,319330000.0,344905000.0,332571250.0
39,327823375.0,4.691762e+06,317295000.0,335197500.0,328377500.0


In [30]:
estadisticas

,mean,std,min,max,median
trial,,,,,
25,337357625.0,4.239044e+06,330357500.0,344245000.0,336985000.0
41,334984375.0,4.716811e+06,327442500.0,347297500.0,334386250.0
26,332557500.0,3.860108e+06,323647500.0,339102500.0,332970000.0
43,332601500.0,5.449226e+06,319330000.0,344905000.0,332571250.0
39,327823375.0,4.691762e+06,317295000.0,335197500.0,328377500.0


##### Carga Modelos y Predice Probabilidades y Ganancias

top_n = 5
top_trials = sorted(
    [t for t in study.trials if t.value is not None],
    key=lambda t: t.value,
    reverse=True
)[:top_n]

trials_top5 = [t.number for t in top_trials]

carpeta_modelos = modelos_path + 'cpt02_modelos_TrainAbr/'
n_semillas = 20

resultados_junio = {}
rows = []

for t in trials_top5:
    for seed in semillas[:n_semillas]:
        model = lgb.Booster(model_file=carpeta_modelos + f"cpt02_00_TrainAbr-TestlJun_Top5Sem_mod_t{t}_s{seed}.txt")
        y_pred_junio = model.predict(X_test)
        resultados_junio[(t, seed)] = y_pred_junio
        rows.append({"trial": t, "seed": seed, "ganancia_junio": gan_prob(y_pred_junio, y_test_binaria)})

df_semillas = pd.DataFrame(rows)

estadisticas = (df_semillas.groupby('trial')['ganancia_junio']
                .agg(['mean', 'std', 'min', 'max', 'median'])
                .sort_values('median', ascending=False))

print(type(resultados_junio), len(resultados_junio))   # dict, 100
print(type(df_semillas), df_semillas.shape)            # DataFrame, (100, 3)
estadisticas

##### Boxplots Ordenados por Mediana de Mayor a Menor

In [31]:
orden_trials = [str(t) for t in estadisticas.index]       # ordenados por mediana, de mayor a menor

fig = go.Figure()

for trial in estadisticas.index:
    datos_trial = df_semillas[df_semillas['trial'] == trial]
    fig.add_trace(go.Box(
        y=datos_trial['ganancia_junio'],
        x=[str(trial)] * len(datos_trial),
        name=str(trial),
        boxpoints=False,
        showlegend=False
    ))

primera_semilla = df_semillas[df_semillas['seed'] == semillas[0]]
fig.add_trace(go.Scatter(
    x=[str(t) for t in primera_semilla['trial']],
    y=primera_semilla['ganancia_junio'],
    mode='markers',
    marker=dict(color='red', size=12, line=dict(color='black', width=1)),
    name=f'semilla {semillas[0]}'
))

fig.update_layout(
    title=f"Ganancia en Junio por modelo ({n_semillas} semillas)",
    xaxis_title="Trial (modelo)",
    yaxis_title="Ganancia en Junio",
    xaxis=dict(categoryorder='array', categoryarray=orden_trials),
    template='plotly_white'
)

fig.show()

##### Desempatando los Top5 con un esquema Público/Privado

In [32]:
# Candidatos empatados
candidatos = [25, 41, 26, 43, 39]

# Lo que vale cada cliente de junio si se lo estimula: +1.072.500 si BAJA+2, -27.500 si no
gan_cliente = np.where(y_test_binaria.values == 1, ganancia_acierto, -costo_estimulo)

# Lista de cada modelo: clientes con prob >= 0.025 (elegida una vez, sobre todo junio)
listas = {(c, s): resultados_junio[(c, s)] >= 0.025
          for c in candidatos for s in semillas[:20]}

# 100 sorteos: 25% público / 75% privado, manteniendo la proporción de BAJA+2
sss = StratifiedShuffleSplit(n_splits=100, test_size=0.25, random_state=semillas[0])

gana_pub  = {c: 0 for c in candidatos}
gana_priv = {c: 0 for c in candidatos}
gan_pub   = {c: [] for c in candidatos}
gan_priv  = {c: [] for c in candidatos}

for idx_priv, idx_pub in sss.split(X_test, y_test_binaria):      # primero el 75%, después el 25%
    for idx, gana, gan in [(idx_pub, gana_pub, gan_pub), (idx_priv, gana_priv, gan_priv)]:
        media = {c: np.mean([gan_cliente[idx][listas[(c, s)][idx]].sum()
                             for s in semillas[:20]]) for c in candidatos}
        gana[max(media, key=media.get)] += 1
        for c in candidatos:
            gan[c].append(media[c])

# Tabla con los dos resultados juntos
tabla_sim = pd.DataFrame({
    'victorias_publico':  gana_pub,
    'victorias_privado':  gana_priv,
    'ganancia_publico_M': {c: round(np.mean(v) / 1e6, 2) for c, v in gan_pub.items()},
    'ganancia_privado_M': {c: round(np.mean(v) / 1e6, 2) for c, v in gan_priv.items()},
})
tabla_sim.index.name = 'trial'
tabla_sim

,victorias_publico,victorias_privado,ganancia_publico_M,ganancia_privado_M
trial,,,,
25,56,93,84.81,252.55
41,20,7,84.25,250.74
26,5,0,83.57,248.99
43,10,0,83.80,248.81
39,9,0,82.60,245.22


#### Ganancia Acumulada

##### BAJA+2 + BAJA+1


**Target Fusionado (BAJA+1 y BAJA+2): Entrenamiento en Abril y Prueba en Junio con el Trial 25**

In [38]:
# Target fusionado: BAJA+1 y BAJA+2 juntos (1) contra CONTINUA (0)
data['clase_binaria_b12'] = np.where(data['clase_ternaria'].isin(['BAJA+1', 'BAJA+2']), 1, 0)

# Abril para entrenar y junio para testear
train_data2 = data[data['foto_mes'] == mes_train]
test_data2 = data[data['foto_mes'] == mes_test]

# Se descartan el target, el auxiliar del BAJA+2 solo, el id y el mes, para que no entren a X
cols_a_descartar = ['clase_ternaria', 'clase_binaria_b2', 'clase_binaria_b12', 'numero_de_cliente', 'foto_mes']

X_train2 = train_data2.drop(columns=cols_a_descartar)
y_train_binaria2 = train_data2['clase_binaria_b12']

X_test2 = test_data2.drop(columns=cols_a_descartar)
y_test_binaria2 = test_data2['clase_binaria_b12']
y_test_class2 = test_data2['clase_ternaria']          # solo para medir con el BAJA+2 real; no entra en X

train_data_lgb2 = lgb.Dataset(X_train2, label=y_train_binaria2)

# Receta del trial 25: sus hiperparámetros y el número de árboles (best_iter) que guardó la optimización
trial_25 = next(t for t in study.trials if t.number == 25)   
best_iter = trial_25.user_attrs['best_iter']
params = {
    'objective': 'binary',
    'boosting_type': 'gbdt',
    'first_metric_only': True,
    'boost_from_average': True,
    'feature_pre_filter': False,
    'max_bin': 31,                                         # fijo de la cátedra
    'min_data_in_leaf': 0,                                 # fijo de la cátedra: el freno es min_sum_hessian_in_leaf
    **trial_25.params,                                     # CAMBIO: 49 -> 25. Los 7 hiperparámetros de la optimización
    'seed': semillas[0],
    'num_threads': lgb_threads,
    'verbose': -1
}

# Entrena con la primera semilla, para esta prueba rápida (las 20 semillas se hacen en la celda de abajo)
model_baja1_2 = lgb.train(
    {**params, 'seed': semillas[0]},
    train_data_lgb2,
    num_boost_round=best_iter
)

# Probabilidades de junio del modelo fusionado
y_pred_baja1_2 = model_baja1_2.predict(X_test2)

# Ganancia con el BAJA+2 real: se acierta solo si el cliente fue BAJA+2, aunque el modelo haya aprendido las dos bajas
def ganancia_real(y_pred, y_true_class, umbral=0.025):
    ganancia = np.where(y_true_class == 'BAJA+2', ganancia_acierto, -costo_estimulo)
    return ganancia[y_pred >= umbral].sum()

ganancia_nueva = ganancia_real(y_pred_baja1_2, y_test_class2.values)
print(f"Ganancia en Junio (target BAJA+1+BAJA+2 vs CONTINUA): {ganancia_nueva:.2f}")

Ganancia en Junio (target BAJA+1+BAJA+2 vs CONTINUA): 370452500.00


**Curva de Ganancia Acumulada en Junio (Target BAJA+1+BAJA+2, Trial 25, una semilla)**

In [40]:
import plotly.graph_objects as go

# Ordena los clientes de junio de mayor a menor probabilidad del modelo fusionado
orden2 = np.argsort(y_pred_baja1_2)[::-1]
# Resultado real de cada cliente en ese orden. Se usa y_test_binaria (solo BAJA+2), no y_test_binaria2, para medir con la ganancia real
ganancia_ordenada2 = np.where(y_test_binaria.values == 1, ganancia_acierto, -costo_estimulo)[orden2]
# Ganancia acumulada: lo que se gana si se estimulan los primeros 1, 2, 3... clientes
ganancia_acumulada2 = np.cumsum(ganancia_ordenada2)

# Mejor corte: el N donde la ganancia acumulada llega a su máximo (+1 porque argmax empieza en 0)
mejor_corte2 = np.argmax(ganancia_acumulada2) + 1
mejor_ganancia2 = ganancia_acumulada2[mejor_corte2 - 1]
# Probabilidad del último cliente que entra en ese corte (en el fusionado la escala es más alta que en el BAJA+2 solo)
p_corte2 = y_pred_baja1_2[orden2][mejor_corte2 - 1]

limite = 20000  # cuantos clientes mostrar en el eje x

fig = go.Figure()

# Curva de ganancia acumulada, hasta el límite del eje
fig.add_trace(go.Scatter(
    x=list(range(1, limite + 1)), y=ganancia_acumulada2[:limite],
    mode='lines', name='Ganancia acumulada', line=dict(color='blue')
))

# Línea vertical punteada en el corte óptimo
fig.add_vline(
    x=mejor_corte2, line_dash='dash', line_color='black',
    annotation_text=f'Corte óptimo: {mejor_corte2} clientes',
    annotation_position='top'
)

# Punto rojo en el máximo, con la probabilidad de corte al lado
fig.add_trace(go.Scatter(
    x=[mejor_corte2], y=[mejor_ganancia2],
    mode='markers+text', marker=dict(color='red', size=10),
    text=[f'p = {p_corte2:.3f}'], textposition='middle right',
    showlegend=False
))

fig.update_layout(
    title='Ganancia acumulada según cantidad de clientes estimulados (target BAJA+1+BAJA+2) - Trial 25',
    xaxis_title='Cantidad de clientes ordenados por probabilidad descendente',
    yaxis_title='Ganancia acumulada',
    xaxis_range=[0, limite],
    template='plotly_white'
)

fig.show()

print(f"Máxima ganancia: {mejor_ganancia2:.2f} en el corte {mejor_corte2}, con probabilidad de corte p = {p_corte2:.4f}")

Máxima ganancia: 380655000.00 en el corte 11158, con probabilidad de corte p = 0.0317


**Fusionado (BAJA+1+BAJA+2): 20 Semillas, Mejor Corte y Ganancia por Corte en Junio (Trial 25)**

In [41]:
n_semillas = 20                                   # cantidad de semillas
cortes = np.arange(4000, 20001, 500)              # cortes fijos (4.000, 4.500, ... 20.000) para el boxplot por corte
carpeta_modelos = modelos_path + 'cpt04_modelos_TrainAbr_FE04b_opt/'

rows = []                                         # ganancia de cada semilla en cada corte fijo
rows_N = []                                       # mejor N y su ganancia, una fila por semilla
resultados_junio_b12 = {}                         # probabilidades de junio de cada modelo fusionado: (trial, semilla) -> probabilidades

for seed in semillas[:n_semillas]:
    params_seed = {**params, 'seed': seed}        # configuración del trial 25, con la semilla de esta vuelta

    # Entrena con abril (target fusionado), guarda el modelo y predice junio
    model_seed = lgb.train(params_seed, train_data_lgb2, num_boost_round=best_iter)
    model_seed.save_model(carpeta_modelos + f"cpt04_TrainAbr-TestlJun-FE04b_Top5Sem_modb12_t25_s{seed}.txt")    
    y_pred_seed = model_seed.predict(X_test2)
    resultados_junio_b12[(25, seed)] = y_pred_seed                                                         

    # Ordena los clientes de junio de mayor a menor probabilidad
    orden_seed = np.argsort(y_pred_seed)[::-1]
    # Resultado real de cada cliente en ese orden, medido con el BAJA+2 real (y_test_binaria): +ganancia si fue BAJA+2, -costo si no
    ganancia_ordenada_seed = np.where(y_test_binaria.values[orden_seed] == 1, ganancia_acierto, -costo_estimulo)
    # Ganancia acumulada: lo que se gana si se estimulan los primeros 1, 2, 3... clientes
    ganancia_acumulada_seed = np.cumsum(ganancia_ordenada_seed)

    # Mejor corte de esta semilla: el N donde la ganancia acumulada llega a su máximo (+1 porque argmax empieza en 0)
    mejor_N = int(np.argmax(ganancia_acumulada_seed)) + 1
    rows_N.append({"seed": seed, "mejor_N": mejor_N, "mejor_ganancia": ganancia_acumulada_seed[mejor_N - 1]})

    # Ganancia de esta semilla en cada corte fijo (-1 porque la posición N está en el índice N-1)
    for corte in cortes:
        rows.append({
            "seed": seed,
            "corte": corte,
            "ganancia": ganancia_acumulada_seed[corte - 1]
        })

# Guarda las probabilidades de los 20 modelos (sirven con el orden de filas fijo que ya tiene data)
with open(modelos_path + 'cpt04_TrainAbr-TestlJun-FE04b_Top5Sem_probb12_t25.pkl', 'wb') as f:                  # CAMBIO: 49 -> 25
    pickle.dump(resultados_junio_b12, f)

# Mejor N por semilla: de acá sale el N central (la mediana) que se pasa a agosto
df_mejorN_b12 = pd.DataFrame(rows_N)
print("Mediana del mejor N (fusionado):", df_mejorN_b12['mejor_N'].median())
print("Mediana de la ganancia máxima:", df_mejorN_b12['mejor_ganancia'].median())

# Ganancia por semilla en cada corte fijo: alimenta las estadísticas y el boxplot
df_ganancia_cortes = pd.DataFrame(rows)

# Estadísticas por corte sobre las 20 semillas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median'])
)
estadisticas_cortes

Mediana del mejor N (fusionado): 14535.0
Mediana de la ganancia máxima: 380270000.0


,mean,std,min,max,median
corte,,,,,
4000,275110000.0,5.481442e+06,264000000,284900000,275000000.0
4500,291170000.0,5.029450e+06,282150000,298650000,290950000.0
5000,307780000.0,3.919264e+06,300300000,314600000,307450000.0
5500,322300000.0,4.464657e+06,315150000,334950000,321200000.0
6000,331760000.0,3.783677e+06,324500000,337700000,332200000.0
6500,338140000.0,4.145118e+06,329450000,344850000,338250000.0
7000,344960000.0,3.593767e+06,338800000,352000000,345400000.0
7500,349580000.0,3.233720e+06,343750000,354750000,349800000.0
8000,354640000.0,4.152792e+06,347600000,361900000,354200000.0


**Boxplot de Ganancia por Corte en Junio (Fusionado, 20 semillas, Trial 25**

In [42]:
# Estadísticas por corte sobre las 20 semillas, más q1 y q3 (percentiles 25 y 75) para las cajas
estadisticas_cortes = (
    df_ganancia_cortes.groupby('corte')['ganancia']
    .agg(['mean', 'std', 'min', 'max', 'median',
          lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
)
estadisticas_cortes.columns = ['mean', 'std', 'min', 'max', 'median', 'q1', 'q3']

fig = go.Figure()

# Un boxplot por corte, armado con las estadísticas ya calculadas
fig.add_trace(go.Box(
    x=[str(c) for c in estadisticas_cortes.index],         # corte como texto: cada uno es una caja separada
    q1=estadisticas_cortes['q1'],                          # borde inferior de la caja
    median=estadisticas_cortes['median'],                  # línea del medio
    q3=estadisticas_cortes['q3'],                          # borde superior de la caja
    lowerfence=estadisticas_cortes['min'],                 # bigote inferior: mínimo de las 20 semillas
    upperfence=estadisticas_cortes['max'],                 # bigote superior: máximo de las 20 semillas
    mean=estadisticas_cortes['mean'],                      # media (línea punteada)
    sd=estadisticas_cortes['std'],
    boxmean=True,
))

fig.update_layout(
    title=f'Distribución de ganancia por corte - Fusionado BAJA+1+BAJA+2 ({n_semillas} semillas) - Trial 25',
    xaxis_title='Cantidad de clientes (corte)',
    yaxis_title='Ganancia acumulada',
    template='plotly_white'
)

fig.show()

**N de Agosto a partir del Mejor Corte de Junio (Fusionado BAJA+1+BAJA+2, Trial 25)**

In [43]:
# Cantidad de clientes de cada mes: de junio (test) y de agosto (el mes a predecir)
clientes_junio = len(y_test_binaria)                          # 164114
clientes_agosto = int((data['foto_mes'] == 202108).sum())     # 164647

# N central en junio: mediana de los 20 mejores N (uno por semilla) del fusionado, y su peso como % de los clientes de junio
N_central_junio = df_mejorN_b12['mejor_N'].median()           # CAMBIO: df_mejorN_b2 -> df_mejorN_b12
pct_central = N_central_junio / clientes_junio

# Barrido: el mismo porcentaje central, corrido -20%, -10%, 0, +10% y +20%
barrido = pd.DataFrame({'variacion': [-0.20, -0.10, 0.0, 0.10, 0.20]})
barrido['pct'] = (pct_central * (1 + barrido['variacion'])).round(4)

# Cada porcentaje se traslada a agosto: N de clientes a estimular en cada submit
barrido['N_agosto'] = (barrido['pct'] * clientes_agosto).round().astype(int)

print(f"Mediana del mejor N en junio (fusionado): {N_central_junio:.0f}  ->  {pct_central*100:.2f}% de {clientes_junio} clientes")
print(f"Clientes en agosto: {clientes_agosto}")
barrido

Mediana del mejor N en junio (fusionado): 14535  ->  8.86% de 164114 clientes
Clientes en agosto: 164647


,variacion,pct,N_agosto
0,-0.2,0.0709,11673
1,-0.1,0.0797,13122
2,0.0,0.0886,14588
3,0.1,0.0974,16037
4,0.2,0.1063,17502


#### Entrenamiento Final: Junio (Target Fusionado) para Predecir Agosto

In [44]:
# Entrenamiento final: junio (con etiquetas) para predecir agosto (sin etiquetas)
mes_train_jun = 202106
mes_pred_ago = 202108

train_jun = data[data['foto_mes'] == mes_train_jun]
pred_ago = data[data['foto_mes'] == mes_pred_ago]

# IDs de los clientes a predecir, guardados ANTES del drop: son los que van al archivo del submit, en el mismo orden que las predicciones
ids_pred_ago = pred_ago['numero_de_cliente'].values

# Mismas columnas descartadas que en abril-junio; el target es el fusionado (cols_a_descartar viene de la celda anterior)
X_train_jun = train_jun.drop(columns=cols_a_descartar)
y_train_jun = train_jun['clase_binaria_b12']
X_pred_ago = pred_ago.drop(columns=cols_a_descartar)

train_data_lgb_jun = lgb.Dataset(X_train_jun, label=y_train_jun)

# Chequeos: mismas columnas que en el entrenamiento de abril (X_train2), 164.647 clientes a predecir, y cuántos positivos hay en junio
print(X_train_jun.shape, X_pred_ago.shape, len(ids_pred_ago))
print((X_train_jun.columns == X_train2.columns).all())
print(y_train_jun.value_counts())

(164114, 599) (164647, 599) 164647
True
clase_binaria_b12
0    162142
1      1972
Name: count, dtype: int64


**Agosto: 20 Semillas Nuevas con la Receta del Trial 34 (Target Fusionado), Entrenadas con Junio**

In [47]:
# 20 semillas nuevas, distintas de las usadas en la validación (semillas[:20]); hacen falta al menos 40 semillas
assert len(semillas) >= 40
semillas_pred = semillas[20:40]

carpeta_modelos_pred = modelos_path + 'cpt04_modelos_TrainJun_FE04b_opt/'
os.makedirs(carpeta_modelos_pred, exist_ok=True)          # carpeta nueva: no se mezcla con los modelos de abril

resultados_pred_b12 = {}                                  # probabilidades de agosto de cada modelo: (trial, semilla) -> probabilidades

for seed in semillas_pred:
    params_seed = {**params, 'seed': seed}                # receta del trial 25 (params y best_iter de la celda del fusionado)

    # Entrena con junio (target fusionado), guarda el modelo y predice agosto
    model_seed = lgb.train(params_seed, train_data_lgb_jun, num_boost_round=best_iter)
    model_seed.save_model(carpeta_modelos_pred + f"cpt04_TrainJun-FE04b_PredAgo_modb12_t25_s{seed}.txt")
    resultados_pred_b12[(25, seed)] = model_seed.predict(X_pred_ago)
    print(f'semilla {seed} lista')

# Guarda las probabilidades y los IDs, en el mismo orden (van juntos para armar los submits)
with open(modelos_path + 'cpt04_TrainJun-FE04b_PredAgo_probb12_t25.pkl', 'wb') as f:
    pickle.dump(resultados_pred_b12, f)
np.save(modelos_path + 'cpt04_TrainJun-FE04b_PredAgo_ids_pred.npy', ids_pred_ago)

print(len(resultados_pred_b12), 'modelos entrenados con junio para predecir agosto')

semilla 964733219 lista
semilla 588265690 lista
semilla 289395646 lista
semilla 979207107 lista
semilla 592528079 lista
semilla 53067711 lista
semilla 799488525 lista
semilla 182671913 lista
semilla 260122723 lista
semilla 447218915 lista
semilla 274781100 lista
semilla 509790657 lista
semilla 651325692 lista
semilla 718509605 lista
semilla 389019099 lista
semilla 699254994 lista
semilla 854151293 lista
semilla 649884351 lista
semilla 408738450 lista
semilla 883777242 lista
20 modelos entrenados con junio para predecir agosto


**Archivos para el bot: 20 semillas por N y semillerío (trial 25, FE04b optimizado)**

In [53]:
# Archivos para el bot: una lista de IDs por cada (N, semilla), más el semillerío de cada N

nro_submit = '04'            # número de submit
meses_sub = 'aja'            # meses (ej. aja = abril-junio-agosto) y otras variantes
fe_sub = 'FE04bOpt'          # dataset + otra información (ej. FE04bOpt = con deltas + optimización)

hacer_20 = False             # True = genera los 20 archivos (uno por semilla) por cada N
hacer_sem = True             # True = genera el semillerío (un archivo) por cada N

variaciones = [-0.1, 0.0, 0.1]          # cortes a generar. Todos: [-0.2, -0.1, 0.0, 0.1, 0.2]. Solo el central: [0.0]
N_central_manual = None                 # N central de agosto a mano (ej. 14,000). None = usa el del barrido
Ns_manual = [12500]                     # lista de N de agosto a mano. None = usa variaciones / N_central_manual

pref_carpeta = f'submit{nro_submit}{meses_sub}{fe_sub}'      # da 'submit04ajaFE04bOpt'
pref_archivo = f'Sbm{nro_submit}{meses_sub}{fe_sub}'         # da 'Sbm04ajaFE04bOpt'

carpeta_subs = dataset_path + 'out/'                    # los submits van en data/out

if Ns_manual is not None:               # CAMBIO: si hay lista a mano, manda esta y se ignora lo demás
    Ns = [int(n) for n in Ns_manual]
elif N_central_manual is None:          # usa el barrido calculado en la celda del N de agosto
    Ns = barrido.loc[barrido['variacion'].round(2).isin(variaciones), 'N_agosto'].tolist()
else:                                   # central a mano; los demás cortes salen de él
    Ns = [int(round(N_central_manual * (1 + v))) for v in variaciones]
print('N a generar:', Ns)               # para verlos antes de escribir

def escribir_ids(ruta, ids):
  # IDs como enteros (nunca notación científica), uno por línea, sin encabezado
    with open(ruta, 'w') as f:
        f.write('\n'.join(str(int(c)) for c in ids))

# Probabilidades de agosto de las 20 semillas, cliente por cliente, en el mismo orden que ids_pred_ago
p_semillas = {seed: resultados_pred_b12[(25, seed)] for seed in semillas_pred}      # 34 -> 25
assert all(len(p) == len(ids_pred_ago) for p in p_semillas.values())

# Semillerío: promedio de las 20 probabilidades, cliente por cliente
p_prom = np.mean(list(p_semillas.values()), axis=0)

for N in Ns:
  # Opción A: 20 archivos, uno por semilla, todos con el mismo N
    if hacer_20:
        carpeta_N = carpeta_subs + f'{pref_carpeta}N{N}/'
        os.makedirs(carpeta_N, exist_ok=True)
        for i, (seed, p) in enumerate(p_semillas.items(), start=1):   # i = 1..20
            idx = np.argsort(p)[::-1][:N]               # los N clientes con mayor probabilidad
            escribir_ids(carpeta_N + f'{pref_archivo}N{N}_s{i:02d}.csv', ids_pred_ago[idx])

  # Opción B: semillerío, un solo archivo por N (carpeta aparte para no mezclarlo con los 20)
    if hacer_sem:
        carpeta_sem = carpeta_subs + f'{pref_carpeta}N{N}Sem/'
        os.makedirs(carpeta_sem, exist_ok=True)
        idx = np.argsort(p_prom)[::-1][:N]
        escribir_ids(carpeta_sem + f'{pref_archivo}N{N}_sem.csv', ids_pred_ago[idx])

# Chequeos antes de mandar: solo de lo que se generó
for N in Ns:
    if hacer_20:
        carpeta_N = carpeta_subs + f'{pref_carpeta}N{N}/'
        archivos = sorted(os.listdir(carpeta_N))
        ok_lineas, ok_e, ok_unicos = True, True, True
        for a in archivos:
            texto = open(carpeta_N + a).read()
            ids = texto.split('\n')
            ok_lineas = ok_lineas and len(ids) == N         # tiene que tener exactamente N líneas
            ok_e = ok_e and ('e+' not in texto)             # sin notación científica
            ok_unicos = ok_unicos and len(set(ids)) == len(ids)   # sin IDs repetidos
        print(N, '| 20 archivos:', len(archivos) == 20, '| N líneas:', ok_lineas, '| sin e+:', ok_e, '| IDs únicos:', ok_unicos)

    if hacer_sem:
        texto = open(carpeta_subs + f'{pref_carpeta}N{N}Sem/' + f'{pref_archivo}N{N}_sem.csv').read()
        ids = texto.split('\n')
        print(N, '| semillerío | N líneas:', len(ids) == N, '| sin e+:', 'e+' not in texto, '| IDs únicos:', len(set(ids)) == len(ids))

N a generar: [12500]
12500 | semillerío | N líneas: True | sin e+: True | IDs únicos: True
